# Combine Cohort A and Cohort B

Unions the finished Cohort A (registry + EHR matched) and Cohort B (EHR-only) feature tables into a
single analytic table, restricted to adults (18+) with a valid utilization cluster assignment. 

In [ ]:
USE CATALOG
 your_catalog;

### Union the two cohorts
Combines `cohort_a_derived.cohort_a_feature_table_edi` (registry-and-EHR-matched cases only)
with `cohort_b_derived.cohort_b_feature_table_edi` (EHR-only), labels each row by cohort, adds
computed age at episode start, and filters to adults with a valid density cluster. Produces
`cohort_a_derived.cohort_a_b_feature_table_edi`, the table used for modeling in
`09_test_train`.

In [ ]:
drop table if exists cohort_a_derived.cohort_a_b_feature_table_edi;
create table cohort_a_derived.cohort_a_b_feature_table_edi as
with one as (select *, 'cohort a reg and ehr' as label from cohort_a_derived.cohort_a_feature_table_edi
where match_type = 'Registry and EHR'

union

select *, 'cohort b ehr only' as label from cohort_b_derived.cohort_b_feature_table_edi),

two as (select o.*, round(date_diff(o.ehr_episode_start, coalesce(b.birth_datetime, a.birth_datetime))/365.25) as age_at_episode
from one o
left join cohort_a.person a on  o.ehr_person_id = a.person_id
left join cohort_b.person b on o.ehr_person_id = b.person_id
)

select * from two
where 
age_at_episode >= 18 
and 
gmm_4_cluster is not null
;